# 01 — Data Understanding

Notebook ini merupakan bagian dari **Fase 1 CRISP-DM** pada penelitian
*Alert Reduction* untuk log Wazuh SOC INSTIKI.

**Tujuan:**
1. Memuat seluruh data log mentah dari arsip `data/raw/`.
2. Melakukan *flattening* dan *cleaning* awal (sesuai `src/transformers.py`).
3. Mengaudit kualitas data: tipe, null, duplikat.
4. Eksplorasi distribusi fitur utama: `agent_name`, `rule_level`, `rule_id`, `decoder_name`, `location`, dan temporal.
5. Menyimpulkan karakteristik dataset untuk mendasari keputusan di fase selanjutnya (Data Preparation).

---
## 1. Setup & Import

In [ ]:
import sys
from pathlib import Path

# Pastikan root project ada di sys.path agar modul src & config dapat diimpor
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from config import (
    DATA_RAW_DIR,
    DATA_PROC_DIR,
    FIGURES_DIR,
    EXTRACT_FIELDS,
    PARQUET_CLEANED,
)
from src.transformers import read_wazuh_logs, flatten_and_clean_logs

# Buat direktori output jika belum ada
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROC_DIR.mkdir(parents=True, exist_ok=True)

# Style visualisasi
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

print("✓ Setup selesai.")
print(f"  Raw data dir  : {DATA_RAW_DIR}")
print(f"  Figures dir   : {FIGURES_DIR}")

---
## 2. Pemuatan Data Mentah

Fungsi `read_wazuh_logs()` membaca semua file `ossec-alerts-*.json.gz`
baik dari dalam arsip `.tar.gz` maupun file loose di `data/raw/`.
Setiap record JSON di-*flatten* sesuai field yang dikonfigurasi di `config.EXTRACT_FIELDS`.

In [ ]:
print("Memulai pembacaan log Wazuh dari arsip...")
print(f"Field yang diekstrak: {list(EXTRACT_FIELDS.keys())}")
print()

raw_records = read_wazuh_logs(DATA_RAW_DIR)
print(f"Total record mentah terbaca: {len(raw_records):,}")

In [ ]:
# Konversi ke DataFrame mentah (sebelum cleaning)
df_raw = pd.DataFrame(raw_records)
print(f"Shape DataFrame mentah: {df_raw.shape}")
print(f"Kolom: {list(df_raw.columns)}")
df_raw.head()

---
## 3. Cleaning & Flattening

Fungsi `flatten_and_clean_logs()` melakukan:
- Drop baris dengan kolom kritis bernilai null (`timestamp`, `agent_name`, `rule_level`, `rule_id`, `rule_groups`, `decoder_name`, `location`).
- Konversi tipe data (`timestamp` → datetime, `rule_level` → int, dll.).
- Normalisasi field `rule_groups` (list → comma-separated string).

In [ ]:
df = flatten_and_clean_logs(raw_records)

rows_removed = len(raw_records) - len(df)
pct_removed  = (rows_removed / len(raw_records)) * 100 if raw_records else 0

print(f"Baris sebelum cleaning : {len(raw_records):,}")
print(f"Baris setelah cleaning : {len(df):,}")
print(f"Baris dihapus          : {rows_removed:,} ({pct_removed:.2f}%)")
print()
df.head()

---
## 4. Audit Kualitas Data

### 4.1 Info & Tipe Data

In [ ]:
df.info()

In [ ]:
print("Tipe data per kolom:")
print(df.dtypes)
print()
print("Statistik deskriptif (kolom numerik):")
df.describe()

### 4.2 Pemeriksaan Null Values

In [ ]:
null_counts = df.isnull().sum()
null_pct    = (df.isnull().mean() * 100).round(2)

null_audit = pd.DataFrame({
    "null_count": null_counts,
    "null_pct (%)": null_pct,
})
print("Audit Null Values:")
print(null_audit.to_string())
print()

if null_counts.sum() == 0:
    print("✓ Tidak ada null values setelah cleaning.")
else:
    print(f"⚠ Terdapat {null_counts.sum()} null values yang perlu ditindaklanjuti.")

### 4.3 Pemeriksaan Baris Duplikat

In [ ]:
n_exact_dup = df.duplicated().sum()
print(f"Jumlah baris duplikat persis (exact duplicate): {n_exact_dup:,}")
print(f"Persentase duplikat: {(n_exact_dup / len(df) * 100):.2f}%")
print()
print("Catatan: Deduplikasi temporal berbasis time-window akan dilakukan")
print("pada fase Data Preparation (notebook 02).")

### 4.4 Kardinalitas Setiap Fitur

In [ ]:
cardinality = df.nunique().reset_index()
cardinality.columns = ["kolom", "unique_count"]
print("Kardinalitas (jumlah nilai unik per kolom):")
print(cardinality.to_string(index=False))

---
## 5. Eksplorasi Distribusi Fitur

### 5.1 Distribusi per Agent (`agent_name`)

In [ ]:
agent_dist = (
    df.groupby("agent_name")
    .size()
    .reset_index(name="alert_count")
    .sort_values("alert_count", ascending=False)
)
print("Distribusi alert per agent:")
print(agent_dist.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=agent_dist, x="agent_name", y="alert_count", ax=ax, color="steelblue")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_title("Distribusi Alert per Agent")
ax.set_xlabel("Agent Name")
ax.set_ylabel("Jumlah Alert")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "distribusi_agent.png", dpi=150)
plt.show()

### 5.2 Distribusi Rule Level (`rule_level`)

In [ ]:
level_dist = (
    df.groupby("rule_level")
    .size()
    .reset_index(name="alert_count")
    .sort_values("rule_level")
)
print("Distribusi alert per rule_level:")
print(level_dist.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=level_dist, x="rule_level", y="alert_count", ax=ax, color="steelblue")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_title("Distribusi Alert per Rule Level")
ax.set_xlabel("Rule Level")
ax.set_ylabel("Jumlah Alert")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "distribusi_rule_level.png", dpi=150)
plt.show()

### 5.3 Distribusi Top 10 Rule ID (`rule_id`)

In [ ]:
rule_dist = (
    df.groupby("rule_id")
    .size()
    .reset_index(name="alert_count")
    .sort_values("alert_count", ascending=False)
)
print("Top 10 Rule ID pemicu alert terbanyak:")
print(rule_dist.head(10).to_string(index=False))

In [ ]:
top10_rules = rule_dist.head(10).copy()

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=top10_rules, x="rule_id", y="alert_count",
    ax=ax, color="steelblue", order=top10_rules["rule_id"]
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_title("Top 10 Rule ID Pemicu Alert Terbanyak")
ax.set_xlabel("Rule ID")
ax.set_ylabel("Jumlah Alert")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "top10_rule_id.png", dpi=150)
plt.show()

### 5.4 Distribusi Decoder (`decoder_name`)

In [ ]:
decoder_dist = (
    df.groupby("decoder_name")
    .size()
    .reset_index(name="alert_count")
    .sort_values("alert_count", ascending=False)
)
print("Distribusi alert per decoder:")
print(decoder_dist.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=decoder_dist, x="decoder_name", y="alert_count",
    ax=ax, color="steelblue", order=decoder_dist["decoder_name"]
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_title("Distribusi Alert per Decoder")
ax.set_xlabel("Decoder Name")
ax.set_ylabel("Jumlah Alert")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "distribusi_decoder.png", dpi=150)
plt.show()

### 5.5 Distribusi Lokasi Sumber Log (`location`)

In [ ]:
location_dist = (
    df.groupby("location")
    .size()
    .reset_index(name="alert_count")
    .sort_values("alert_count", ascending=False)
)
print("Distribusi alert berdasarkan lokasi sumber log:")
print(location_dist.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=location_dist, x="location", y="alert_count",
    ax=ax, color="steelblue", order=location_dist["location"]
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_title("Distribusi Alert per Lokasi Sumber Log")
ax.set_xlabel("Location")
ax.set_ylabel("Jumlah Alert")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "distribusi_location.png", dpi=150)
plt.show()

### 5.6 Distribusi Top 10 Rule Description (`rule_description`)

In [ ]:
desc_dist = (
    df.groupby("rule_description")
    .size()
    .reset_index(name="alert_count")
    .sort_values("alert_count", ascending=False)
)
print("Top 10 Rule Description pemicu alert terbanyak:")
print(desc_dist.head(10).to_string(index=False))

---
## 6. Eksplorasi Temporal

### 6.1 Distribusi Alert per Hari

In [ ]:
df["_date"] = df["timestamp"].dt.date
daily = df.groupby("_date").size().reset_index(name="alert_count")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily["_date"], daily["alert_count"], marker="o", markersize=3, linewidth=1.2, color="steelblue")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_title("Distribusi Alert per Hari (Data Mentah)")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Jumlah Alert")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "distribusi_harian_raw.png", dpi=150)
plt.show()

### 6.2 Distribusi Alert per Jam

In [ ]:
df["_hour"] = df["timestamp"].dt.hour
hourly = df.groupby("_hour").size().reset_index(name="alert_count")

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=hourly, x="_hour", y="alert_count", ax=ax, color="steelblue")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_title("Distribusi Alert per Jam (Data Mentah)")
ax.set_xlabel("Jam (0-23)")
ax.set_ylabel("Jumlah Alert")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "distribusi_jam_raw.png", dpi=150)
plt.show()

### 6.3 Distribusi per Bulan

In [ ]:
df["_month"] = df["timestamp"].dt.to_period("M").astype(str)
monthly = (
    df.groupby("_month")
    .size()
    .reset_index(name="alert_count")
    .sort_values("_month")
)
print("Jumlah alert per bulan:")
print(monthly.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=monthly, x="_month", y="alert_count", ax=ax, color="steelblue")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_title("Jumlah Alert per Bulan")
ax.set_xlabel("Bulan")
ax.set_ylabel("Jumlah Alert")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "tren_alert_per_bulan.png", dpi=150)
plt.show()

### 6.4 Heatmap Alert: Hari × Jam

In [ ]:
df["_weekday"] = df["timestamp"].dt.day_name()

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
pivot = df.pivot_table(index="_weekday", columns="_hour", aggfunc="size", fill_value=0)
pivot = pivot.reindex(day_order)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot, cmap="YlOrRd", annot=False, fmt=",d", linewidths=0.3, ax=ax)
ax.set_title("Heatmap Alert: Hari × Jam")
ax.set_xlabel("Jam")
ax.set_ylabel("Hari")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "heatmap_hari_jam.png", dpi=150)
plt.show()

---
## 7. Ringkasan Temuan Data Understanding

In [ ]:
# Bersihkan kolom temporer
df.drop(columns=["_date", "_hour", "_month", "_weekday"], inplace=True, errors="ignore")

print("=" * 60)
print("RINGKASAN DATA UNDERSTANDING")
print("=" * 60)
print(f"Total record mentah              : {len(raw_records):,}")
print(f"Total record setelah cleaning    : {len(df):,}")
print(f"Record tereliminasi              : {rows_removed:,} ({pct_removed:.2f}%)")
print(f"Jumlah kolom                     : {len(df.columns)}")
print(f"Kolom                            : {list(df.columns)}")
print(f"Null values setelah cleaning     : {df.isnull().sum().sum()}")
print(f"Duplikat persis                  : {n_exact_dup:,}")
print(f"Jumlah agent unik               : {df['agent_name'].nunique()}")
print(f"Jumlah rule_id unik             : {df['rule_id'].nunique()}")
print(f"Jumlah decoder unik             : {df['decoder_name'].nunique()}")
print(f"Rentang waktu                    : {df['timestamp'].min()} s.d. {df['timestamp'].max()}")
print("=" * 60)
print()
print("✓ Fase Data Understanding selesai.")
print("  Lanjutkan ke notebook 02_data-preparation.ipynb untuk")
print("  feature engineering, deduplikasi temporal, dan persiapan")
print("  data untuk fase modeling.")